# Análisis Automático de Preguntas Abiertas

## Descripción del Notebook
Este notebook proporciona herramientas para procesar y analizar respuestas de texto libre (preguntas abiertas) utilizando Procesamiento del Lenguaje Natural (NLP) con spaCy.

## Tecnología Utilizada
- **spaCy**: Librería avanzada de NLP en Python
- **TF-IDF**: Vectorización de texto basada en frecuencia de términos
- **Similitud Coseno**: Medida de similitud entre textos

## Metodología
1. **Lematización**: Reducir palabras a su forma base
2. **Vectorización TF-IDF**: Convertir texto a representación numérica
3. **Similitud Coseno**: Comparar cada respuesta con referencia
4. **Clasificación Temática**: Categorizar automáticamente respuestas

## Aplicaciones Prácticas
- Análisis de encuestas y feedback
- Procesamiento de comentarios de clientes
- Clasificación automática de respuestas educativas
- Detección de temas principales en corpus de texto


In [ ]:
import os
os.system('pip install spacy scikit-learn pandas openpyxl matplotlib seaborn ipywidgets -q')

import spacy
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
import warnings

warnings.filterwarnings('ignore')

# Configuración de visualización
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Librerías cargadas exitosamente")


In [ ]:
# Cargar modelo de spaCy con manejo de errores
print("Cargando modelo de lenguaje en español (spaCy)...")

try:
    nlp = spacy.load("es_core_news_sm")
    print("✓ Modelo 'es_core_news_sm' cargado exitosamente.")
except OSError:
    print("⚠ Modelo no encontrado. Descargando...")
    import subprocess
    subprocess.run([
        "python", "-m", "spacy", "download", "es_core_news_sm"
    ], capture_output=True)
    nlp = spacy.load("es_core_news_sm")
    print("✓ Modelo descargado e instalado.")

# Configuración
REFERENCIA_QUESTION = "¿Cuáles son los problemas más importantes, urgentes o prioritarios de atender?"
MAX_ROWS = 8000


In [ ]:
def lemmatizer(text):
    """Extrae lemmas de sustantivos, adjetivos y verbos.

    Procesa un texto usando spaCy y devuelve una lista de lemmas
    excluyendo stop words y signos de puntuación. Solo mantiene
    palabras que sean sustantivos (NOUN), adjetivos (ADJ) o verbos (VERB).

    Args:
        text (str): Texto a procesar.

    Returns:
        list: Lista de lemmas en minúsculas.

    Example:
        >>> lemmatizer("Los gatos corren rápidamente")
        ['gato', 'correr', 'rápido']
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return []

    try:
        doc = nlp(text)
        lemmas = [
            token.lemma_.lower() for token in doc
            if token.pos_ in ['NOUN', 'ADJ', 'VERB']
            and not token.is_stop
            and not token.is_punct
        ]
        return lemmas
    except Exception as e:
        print(f"Error procesando texto: {e}")
        return []


def standardize_category(text):
    """Clasifica automáticamente texto en categorías predefinidas.

    Utiliza expresiones regulares para detectar palabras clave
    y asignar categorías temáticas. Es case-insensitive.

    Args:
        text (str): Texto a clasificar.

    Returns:
        str: Categoría asignada o el texto sin clasificar.

    Categories:
        - Seguridad Ciudadana
        - Costo de Vida / Economía
        - Corrupción / Política
        - Desempleo
        - Infraestructura Vial
        - Educación
        - Salud Pública
        - Medio Ambiente
    """
    if pd.isna(text) or text == "" or not isinstance(text, str):
        return None

    text_lower = text.lower().strip()

    # Definir patrones por categoría
    patterns = {
        "Seguridad Ciudadana": r"seguridad|crimen|delincuencia|violencia|inseguridad|pandilla|narcotráfico|robo",
        "Costo de Vida / Economía": r"vida cara|costo|inflación|pobreza|dinero|caro|economía|canasta|precio",
        "Corrupción / Política": r"corrupción|autoritarismo|impunidad|chorizo|malversación|político",
        "Desempleo": r"empleo|desempleo|trabajo|despido|oportunidades|desocupación|laboral",
        "Infraestructura Vial": r"transporte|carretera|tráfico|infraestructura|vial|presa|caos vial|ruta",
        "Educación": r"educación|escuela|colegio|universidad|primaria|secundaria|enseñanza",
        "Salud Pública": r"salud|caja|ccss|hospital|ebais|médica|servicios de salud|clínica",
        "Medio Ambiente": r"ambiente|agua|contaminación|basura|río|clima|residuos|medio ambiente|ecología",
    }

    for category, pattern in patterns.items():
        if re.search(pattern, text_lower):
            return category

    return text.title()  # Sin categoría


def process_texts(df, text_column):
    """Procesa un DataFrame de textos y calcula métricas de similitud.

    Realiza:
    1. Vectorización TF-IDF
    2. Cálculo de similitud coseno respecto a texto referencia
    3. Extracción de palabras clave principales

    Args:
        df (pd.DataFrame): DataFrame con textos
        text_column (str): Nombre de columna con textos

    Returns:
        pd.DataFrame: DataFrame extendido con columnas:
            - Puntaje_Relevancia
            - Propuesta_1, Propuesta_2, Propuesta_3

    Raises:
        ValueError: Si la columna no existe
        TypeError: Si df no es DataFrame
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("El primer argumento debe ser un pandas DataFrame")

    if text_column not in df.columns:
        raise ValueError(f"Columna '{text_column}' no existe en DataFrame")

    print(f"Procesando {len(df)} textos...")

    # Limpiar textos
    textos_validos = df[text_column].astype(str).fillna('').str.strip()
    corpus = [REFERENCIA_QUESTION] + textos_validos.tolist()

    print("  - Vectorizando con TF-IDF...")
    vectorizer = TfidfVectorizer(
        tokenizer=lemmatizer,
        min_df=1,
        max_df=0.95,
        lowercase=True
    )
    tfidf_matrix = vectorizer.fit_transform(corpus)

    print("  - Calculando similitud coseno...")
    similitud = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:])

    print("  - Extrayendo palabras clave...")
    feature_names = np.array(vectorizer.get_feature_names_out())
    propuestas = []

    for i, row in enumerate(tfidf_matrix[1:]):
        top_indices = np.argsort(row.toarray()).flatten()[::-1]
        top_keywords = [
            feature_names[idx] for idx in top_indices
            if feature_names[idx].strip() != ''
        ]
        top_3 = (top_keywords[:3] + [None, None, None])[:3]
        propuestas.append(top_3)

    df_propuestas = pd.DataFrame(
        propuestas,
        columns=['Propuesta_1', 'Propuesta_2', 'Propuesta_3'],
        index=df.index
    )

    # Combinar resultados
    df_resultado = df.copy()
    df_resultado['Puntaje_Relevancia'] = similitud.flatten()

    for col in df_propuestas.columns:
        df_resultado[col] = df_propuestas[col]

    print("✓ Procesamiento completado.")
    return df_resultado


In [ ]:
# OPCIÓN 1: Cargar desde archivo local
print("=" * 70)
print("CARGA DE DATOS")
print("=" * 70)
print("""
Para ejecutar este notebook LOCALMENTE:
1. Coloca tu archivo Excel en el mismo directorio
2. Reemplaza 'tu_archivo.xlsx' con el nombre real
3. Ejecuta esta celda

Para Google Colab:
- Usa el widget de carga de archivos que aparecerá abajo
""")

# Define aquí la ruta de tu archivo
LOCAL_FILE = 'datos_respuestas.xlsx'

try:
    if os.path.exists(LOCAL_FILE):
        print(f"Leyendo {LOCAL_FILE}...")
        df_cargado = pd.read_excel(LOCAL_FILE)
        print(f"✓ Archivo cargado: {len(df_cargado)} filas")
        print(f"\nColumnas disponibles: {list(df_cargado.columns)}")
        print(f"\nPrimeras filas:")
        display(df_cargado.head())
    else:
        print(f"⚠ Archivo '{LOCAL_FILE}' no encontrado")
        print("Por favor, especifica la ruta correcta o carga un archivo")
        df_cargado = None
except Exception as e:
    print(f"✗ Error al leer archivo: {e}")
    df_cargado = None


In [ ]:
# EJEMPLO DE ANÁLISIS
# Si no tienes un archivo, usa estos datos de ejemplo

ejemplo_textos = {
    'Pregunta': ['¿Cuál es el principal problema?'] * 5,
    'Respuesta': [
        "La inseguridad y la delincuencia son graves problemas que afectan nuestra comunidad",
        "El desempleo y la falta de oportunidades de trabajo",
        "La contaminación ambiental y la falta de agua limpia",
        "Los servicios de salud son inadecuados y muy costosos",
        "La educación necesita más presupuesto y mejores infraestructuras"
    ]
}

df_ejemplo = pd.DataFrame(ejemplo_textos)

print("Datos de ejemplo cargados")
print(f"Tamaño: {len(df_ejemplo)} respuestas")
display(df_ejemplo)


In [ ]:
def analizar_respuestas(df, columna_respuesta, umbral_relevancia=0.25):
    """Función principal de análisis.

    Args:
        df (pd.DataFrame): DataFrame con respuestas
        columna_respuesta (str): Nombre de columna con textos
        umbral_relevancia (float): Umbral de similitud (0-1)

    Returns:
        pd.DataFrame: Resultados con clasificaciones
    """

    # Procesar textos
    df_procesado = process_texts(df, columna_respuesta)

    # Aplicar umbral de relevancia
    df_procesado['Es_Relevante'] = df_procesado['Puntaje_Relevancia'] >= umbral_relevancia

    # Clasificar proposiciones
    df_procesado['Clasificacion_1'] = df_procesado['Propuesta_1'].apply(standardize_category)
    df_procesado['Clasificacion_2'] = df_procesado['Propuesta_2'].apply(standardize_category)
    df_procesado['Clasificacion_3'] = df_procesado['Propuesta_3'].apply(standardize_category)

    return df_procesado


# Ejecutar análisis en datos de ejemplo
print("\nIniciando análisis de ejemplo...")
print("=" * 70)

df_resultados = analizar_respuestas(
    df_ejemplo,
    columna_respuesta='Respuesta',
    umbral_relevancia=0.20
)

print("\nRESULTADOS DEL ANÁLISIS:")
print("-" * 70)

# Estadísticas generales
total = len(df_resultados)
relevantes = df_resultados['Es_Relevante'].sum()

print(f"Total de respuestas: {total}")
print(f"Relevantes: {relevantes}")
print(f"Irrelevantes: {total - relevantes}")

print("\nPuntajes de Relevancia:")
print(df_resultados['Puntaje_Relevancia'].describe())

print("\nCategorías Identificadas:")
categorias = pd.concat([
    df_resultados['Clasificacion_1'],
    df_resultados['Clasificacion_2'],
    df_resultados['Clasificacion_3']
]).dropna().value_counts()
print(categorias)

# Visualizaciones
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de relevancia
axes[0].hist(df_resultados['Puntaje_Relevancia'], bins=20, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(0.25, color='red', linestyle='--', linewidth=2, label='Umbral')
axes[0].set_title('Distribución de Puntajes de Relevancia')
axes[0].set_xlabel('Puntaje de Relevancia')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Top categorías
top_categorias = categorias.nlargest(8)
axes[1].barh(range(len(top_categorias)), top_categorias.values, color='coral', alpha=0.8)
axes[1].set_yticks(range(len(top_categorias)))
axes[1].set_yticklabels(top_categorias.index)
axes[1].set_title('Problemas Identificados (Top 8)')
axes[1].set_xlabel('Frecuencia')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n✓ Análisis completado")


## Ejercicios para el Estudiante

### Ejercicio 1: Análisis Temático Completo
1. Carga un archivo Excel con 50+ respuestas abiertas
2. Ejecuta el análisis con diferentes umbrales de relevancia (0.15, 0.25, 0.35)
3. ¿Cómo cambian los resultados según el umbral?
4. ¿Cuál es el umbral más apropiado para tus datos?

### Ejercicio 2: Validar Clasificaciones
1. Revisa manualmente 10 respuestas y sus clasificaciones automáticas
2. ¿Cuán precisa es la clasificación?
3. Identifica casos problemáticos
4. ¿Qué palabras clave se necesitarían para mejorar?

### Ejercicio 3: Análisis de Palabras Clave
1. Extrae palabras clave por categoría
2. Visualiza nubes de palabras por tema
3. Identifica palabras únicas por categoría
4. ¿Hay términos que aparecen en múltiples categorías?

---

**Notas**:
- Este análisis funciona mejor con 100+ respuestas
- La calidad depende de la precisión de los patrones regex
- Puedes extender con modelos de ML más avanzados
